# 第一部分 数据获取

## 1. 建立项目文件夹

In [ ]:
import os

# 当前文件夹作为项目根目录
root = os.getcwd()  # 这里就是 dshw-p01

# 目录结构
dirs = [
    f"{root}/data",
    f"{root}/data/stock",
    f"{root}/data/index",
    f"{root}/data/macro",
    f"{root}/data/finance",
    f"{root}/data/clean",
    f"{root}/data/combined",
    f"{root}/output",
]

# 文件列表
files = [
    f"{root}/README.md",
    f"{root}/report.html",
    f"{root}/requirements.txt",
    f"{root}/.gitignore",
    f"{root}/01_download.ipynb",
    f"{root}/02_clean.ipynb",
    f"{root}/03_analysis.ipynb",
    f"{root}/download_log.txt",
]

# 创建目录
for d in dirs:
    os.makedirs(d, exist_ok=True)

# 创建文件
for f in files:
    if not os.path.exists(f):
        open(f, "w", encoding="utf-8").close()


## 2. 下载股票过去 5 年（2020-01-01 至今）的后复权日度行情

In [21]:
import baostock as bs
import pandas as pd
from datetime import datetime
import os

# ---------------------------
# 配置
# ---------------------------
LOG_FILE = "download_log.txt"
DATA_FOLDER = "data/stock"
os.makedirs(DATA_FOLDER, exist_ok=True)

def log_message(status, code, msg):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{now}] [{status}] {code} - {msg}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# 登录
lg = bs.login()
if lg.error_code != "0":
    log_message("FAILED", "SYSTEM", f"Baostock 登录失败: {lg.error_msg}")
else:
    log_message("SUCCESS", "SYSTEM", "Baostock 登录成功")

# ---------------------------
# 股票列表
# ---------------------------
stocks = [
    ("600036", "招商银行"),
    ("601166", "兴业银行"),
    ("600519", "贵州茅台"),
    ("000858", "五粮液"),
    ("600104", "上汽集团"),
    ("002594", "比亚迪"),
    ("600048", "保利发展"),
    ("000002", "万科A"),
    ("601857", "中国石油"),
    ("600050", "中国联通"),
]

start_date = "2020-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")
adjustflag = "3"  # 复权：0-不复权，1-前复权，2-后复权，3-不复权

# ---------------------------
# 下载股票日线
# ---------------------------
for code, name in stocks:
    bs_code = "sh." + code if code.startswith("6") else "sz." + code
    try:
        rs = bs.query_history_k_data_plus(
            bs_code,
            "date,code,open,high,low,close,preclose,volume,amount,adjustflag",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag=adjustflag
        )
        if rs.error_code != "0":
            raise Exception(rs.error_msg)

        data_list = []
        while rs.next():
            data_list.append(rs.get_row_data())

        if not data_list:
            log_message("FAILED", code, f"{name} 下载失败: 无数据返回")
            continue

        df = pd.DataFrame(data_list, columns=rs.fields)
        df.to_csv(f"{DATA_FOLDER}/stock_{code}.csv", index=False, encoding="utf-8-sig")
        log_message("SUCCESS", code, f"{name} 下载成功, 共 {len(df)} 行")

    except Exception as e:
        log_message("FAILED", code, f"{name} 下载失败: {e}")

# 登出
bs.logout()
log_message("INFO", "SYSTEM", "Baostock 登出完成")

login success!
[2026-04-08 18:55:12] [SUCCESS] SYSTEM - Baostock 登录成功
[2026-04-08 18:55:16] [SUCCESS] 600036 - 招商银行 下载成功, 共 1516 行
[2026-04-08 18:55:18] [SUCCESS] 601166 - 兴业银行 下载成功, 共 1516 行
[2026-04-08 18:55:21] [SUCCESS] 600519 - 贵州茅台 下载成功, 共 1516 行
[2026-04-08 18:55:24] [SUCCESS] 000858 - 五粮液 下载成功, 共 1516 行
[2026-04-08 18:55:26] [SUCCESS] 600104 - 上汽集团 下载成功, 共 1516 行
[2026-04-08 18:55:28] [SUCCESS] 002594 - 比亚迪 下载成功, 共 1516 行
[2026-04-08 18:55:32] [SUCCESS] 600048 - 保利发展 下载成功, 共 1516 行
[2026-04-08 18:55:34] [SUCCESS] 000002 - 万科A 下载成功, 共 1516 行
[2026-04-08 18:55:37] [SUCCESS] 601857 - 中国石油 下载成功, 共 1516 行
[2026-04-08 18:55:41] [SUCCESS] 600050 - 中国联通 下载成功, 共 1516 行
logout success!
[2026-04-08 18:55:41] [INFO] SYSTEM - Baostock 登出完成


## 3. 下载市场指数

In [23]:
import baostock as bs
import pandas as pd
from datetime import datetime
import os

# ---------------------------
# 初始化
# ---------------------------
LOG_FILE = "download_log.txt"
DATA_FOLDER = "data/index"
os.makedirs(DATA_FOLDER, exist_ok=True)

def log_message(status, code, msg):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{now}] [{status}] {code} - {msg}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# 登录
lg = bs.login()
if lg.error_code != "0":
    log_message("FAILED", "SYSTEM", f"Baostock 登录失败: {lg.error_msg}")
else:
    log_message("SUCCESS", "SYSTEM", "Baostock 登录成功")

# ---------------------------
# 指数列表
# ---------------------------
indices = [
    ("sh.000300", "沪深 300"),
    ("sh.000905", "中证 500")  # 自选，理由：中小盘股票波动性较大，可对比沪深300
]

start_date = "2020-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")

# ---------------------------
# 下载指数日线
# ---------------------------
for code, name in indices:
    try:
        rs = bs.query_history_k_data_plus(
            code,
            "date,code,open,high,low,close,preclose,volume,amount",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag="3"  # 指数通常不复权
        )
        if rs.error_code != "0":
            raise Exception(rs.error_msg)

        data_list = []
        while rs.next():
            data_list.append(rs.get_row_data())
        if not data_list:
            raise Exception("无数据返回")

        df = pd.DataFrame(data_list, columns=rs.fields)
        filename = code.split(".")[1]  # 例如 000300
        df.to_csv(f"{DATA_FOLDER}/index_{filename}.csv", index=False, encoding="utf-8-sig")
        log_message("SUCCESS", code, f"{name} 下载完成, 共 {len(df)} 行")

    except Exception as e:
        log_message("FAILED", code, f"{name} 下载失败: {e}")

# ---------------------------
# 登出
# ---------------------------
bs.logout()
log_message("INFO", "SYSTEM", "Baostock 登出完成")

login success!
[2026-04-08 18:56:00] [SUCCESS] SYSTEM - Baostock 登录成功
[2026-04-08 18:56:03] [SUCCESS] sh.000300 - 沪深 300 下载完成, 共 1516 行
[2026-04-08 18:56:06] [SUCCESS] sh.000905 - 中证 500 下载完成, 共 1516 行
logout success!
[2026-04-08 18:56:07] [INFO] SYSTEM - Baostock 登出完成


## 4. 下载月度宏观指标

In [26]:
import akshare as ak
import pandas as pd
from datetime import datetime
import os

# ---------------------------
# 配置
# ---------------------------
LOG_FILE = "download_log.txt"
DATA_FOLDER = "data/macro"
os.makedirs(DATA_FOLDER, exist_ok=True)

def log_message(status, code, msg):
    """日志函数"""
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{now}] [{status}] {code} - {msg}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# ---------------------------
# CPI 同比增速
# ---------------------------
try:
    cpi = ak.macro_china_cpi_yearly()
    if cpi is None or cpi.empty:
        log_message("FAILED", "CPI", "CPI 数据为空")
    else:
        file_path = os.path.join(DATA_FOLDER, "macro_cpi.csv")
        cpi.to_csv(file_path, index=False, encoding="utf-8-sig")
        log_message("SUCCESS", "CPI", f"CPI 下载完成, 共 {len(cpi)} 行, 保存到 {file_path}")
except Exception as e:
    log_message("FAILED", "CPI", f"CPI 下载失败: {e}")

# ---------------------------
# 人民币/美元汇率
# ---------------------------
try:
    usd = ak.currency_boc_sina(symbol="美元")
    if usd is None or usd.empty:
        log_message("FAILED", "USD", "美元汇率数据为空")
    else:
        file_path = os.path.join(DATA_FOLDER, "macro_usd.csv")
        usd.to_csv(file_path, index=False, encoding="utf-8-sig")
        log_message("SUCCESS", "USD", f"美元汇率下载完成, 共 {len(usd)} 行, 保存到 {file_path}")
except Exception as e:
    log_message("FAILED", "USD", f"美元汇率下载失败: {e}")

[2026-04-08 19:00:01] [SUCCESS] CPI - CPI 下载完成, 共 477 行, 保存到 data/macro\macro_cpi.csv


[2026-04-08 19:00:02] [SUCCESS] USD - 美元汇率下载完成, 共 180 行, 保存到 data/macro\macro_usd.csv


## 5. 查看目标财务指标位置名称

In [31]:
import baostock as bs
import pandas as pd

bs.login()

stock_code = "600036"  # 招商银行示例
bs_code = f"sh.{stock_code}" if stock_code.startswith("6") else f"sz.{stock_code}"

# 测试利润表
rs_profit = bs.query_profit_data(code=bs_code, year="2023", quarter="4")
print("利润表字段:", rs_profit.fields)
data = []
while rs_profit.next():
    data.append(rs_profit.get_row_data())
df_profit = pd.DataFrame(data, columns=rs_profit.fields)
print(df_profit.head())

# 测试资产负债表
rs_balance = bs.query_balance_data(code=bs_code, year="2023", quarter="4")
print("资产负债表字段:", rs_balance.fields)
data = []
while rs_balance.next():
    data.append(rs_balance.get_row_data())
df_balance = pd.DataFrame(data, columns=rs_balance.fields)
print(df_balance.head())

bs.logout()

login success!
利润表字段: ['code', 'pubDate', 'statDate', 'roeAvg', 'npMargin', 'gpMargin', 'netProfit', 'epsTTM', 'MBRevenue', 'totalShare', 'liqaShare']
        code     pubDate    statDate    roeAvg  npMargin gpMargin  \
0  sh.600036  2024-03-26  2023-12-31  0.145016  0.436438            

             netProfit    epsTTM            MBRevenue      totalShare  \
0  148006000000.000000  5.812962  339123000000.000000  25219845601.00   

        liqaShare  
0  20628944429.00  
资产负债表字段: ['code', 'pubDate', 'statDate', 'currentRatio', 'quickRatio', 'cashRatio', 'YOYLiability', 'liabilityToAsset', 'assetToEquity']
        code     pubDate    statDate currentRatio quickRatio cashRatio  \
0  sh.600036  2024-03-26  2023-12-31                                     

  YOYLiability liabilityToAsset assetToEquity  
0     0.082538         0.901552     10.157676  
logout success!


## 6. 获取股票最近5个年度的财务指标

In [32]:
import baostock as bs
import pandas as pd
import os
import time
from datetime import datetime

# ---------------------------
project_root = "."
data_folder = os.path.join(project_root, "data/finance")
log_file = os.path.join(project_root, "download_log.txt")
os.makedirs(data_folder, exist_ok=True)

def log(status, code, msg):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{now}] [{status}] {code} - {msg}"
    print(line)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# ---------------------------
stocks = [
    {"code": "600036", "name": "招商银行"},
    {"code": "601166", "name": "兴业银行"},
    {"code": "600519", "name": "贵州茅台"},
    {"code": "000858", "name": "五粮液"},
    {"code": "600104", "name": "上汽集团"},
    {"code": "002594", "name": "比亚迪"},
    {"code": "600048", "name": "保利发展"},
    {"code": "000002", "name": "万科A"},
    {"code": "601857", "name": "中国石油"},
    {"code": "600050", "name": "中国联通"},
]

# ---------------------------
bs.login()
records = []

years = range(2019, 2024)  # 最近 5 年

for stock in stocks:
    code = stock["code"]
    name = stock["name"]
    bs_code = f"sz.{code}" if code.startswith("0") else f"sh.{code}"
    total_count = 0

    for year in years:
        # ---- 利润表 ----
        try:
            rs = bs.query_profit_data(code=bs_code, year=str(year), quarter="4")
            if rs.error_code != "0":
                log("FAILED", code, f"{name} {year}Q4 利润表下载失败: {rs.error_msg}")
            else:
                while rs.next():
                    row = rs.get_row_data()
                    df = pd.DataFrame([row], columns=rs.fields)
                    roe = df['roeAvg'].iloc[0]
                    np_margin = df['npMargin'].iloc[0]
                    if roe != "" and roe is not None:
                        records.append({
                            "code": code, "name": name,
                            "year": year, "indicator": "ROE", "value": float(roe)*100
                        })
                        total_count += 1
                    if np_margin != "" and np_margin is not None:
                        records.append({
                            "code": code, "name": name,
                            "year": year, "indicator": "净利润率", "value": float(np_margin)*100
                        })
                        total_count += 1
        except Exception as e:
            log("FAILED", code, f"{name} {year}Q4 利润表异常: {e}")

        time.sleep(0.2)

        # ---- 资产负债表 ----
        try:
            rs = bs.query_balance_data(code=bs_code, year=str(year), quarter="4")
            if rs.error_code != "0":
                log("FAILED", code, f"{name} {year}Q4 资产负债表下载失败: {rs.error_msg}")
            else:
                while rs.next():
                    row = rs.get_row_data()
                    df = pd.DataFrame([row], columns=rs.fields)
                    debt = df['liabilityToAsset'].iloc[0]
                    if debt != "" and debt is not None:
                        records.append({
                            "code": code, "name": name,
                            "year": year, "indicator": "资产负债率", "value": float(debt)*100
                        })
                        total_count += 1
        except Exception as e:
            log("FAILED", code, f"{name} {year}Q4 资产负债表异常: {e}")

    log("SUCCESS", code, f"{name} 数据提取完成, 共 {total_count} 条记录")

# 保存 CSV
if records:
    df = pd.DataFrame(records)
    df.to_csv(os.path.join(data_folder, "finance_ratios.csv"), index=False, encoding="utf-8-sig")
    log("INFO", "SYSTEM", f"财务数据保存完成, 共 {len(df)} 条记录")
else:
    log("ERROR", "SYSTEM", "未获取到任何财务数据")

bs.logout()
log("INFO", "SYSTEM", "Baostock 登出完成")

login success!
[2026-04-08 19:05:21] [SUCCESS] 600036 - 招商银行 数据提取完成, 共 15 条记录
[2026-04-08 19:05:45] [SUCCESS] 601166 - 兴业银行 数据提取完成, 共 15 条记录
[2026-04-08 19:06:07] [SUCCESS] 600519 - 贵州茅台 数据提取完成, 共 15 条记录
[2026-04-08 19:06:31] [SUCCESS] 000858 - 五粮液 数据提取完成, 共 15 条记录
[2026-04-08 19:06:51] [SUCCESS] 600104 - 上汽集团 数据提取完成, 共 15 条记录
[2026-04-08 19:07:14] [SUCCESS] 002594 - 比亚迪 数据提取完成, 共 15 条记录
[2026-04-08 19:07:40] [SUCCESS] 600048 - 保利发展 数据提取完成, 共 15 条记录
[2026-04-08 19:08:06] [SUCCESS] 000002 - 万科A 数据提取完成, 共 15 条记录
[2026-04-08 19:08:31] [SUCCESS] 601857 - 中国石油 数据提取完成, 共 15 条记录
[2026-04-08 19:08:52] [SUCCESS] 600050 - 中国联通 数据提取完成, 共 15 条记录
[2026-04-08 19:08:52] [INFO] SYSTEM - 财务数据保存完成, 共 150 条记录
logout success!
[2026-04-08 19:08:53] [INFO] SYSTEM - Baostock 登出完成
